# Project 1 – Traffic Light Object Detection

Train, validate, and run inference for the traffic-light colour detector using the
command line interface shipped with this repository. Execute each cell in order to
reproduce the full workflow on your machine.

- **Dataset root**: `Small Traffic Light.v1i.yolov11`
- **Classes**: green, off, red, wait_on, yellow
- **Key files**:
  - `Project_1_object_detection_traffic_light.py` – primary CLI entrypoint
  - `Small Traffic Light.v1i.yolov11/` – YOLO-format dataset
  - `runs/` & `outputs/` – generated training and inference artefacts

In [ ]:
import sys, subprocess

commands = [
    [sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"],
    [sys.executable, "-m", "pip", "install", "ultralytics", "opencv-python", "pandas", "numpy", "tqdm", "nbformat", "matplotlib", "pyyaml"],
    [sys.executable, "-m", "pip", "install", "sahi", "supervision"],
]
for cmd in commands:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=False)

import json, os, random, sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
from pathlib import Path

DATA_ROOT = Path("Small Traffic Light.v1i.yolov11")
EPOCHS = 120
IMGSZ = 640
BATCH = 16
DEVICE = "auto"
SEED = 42
USE_SAHI = True
VIDEO_PATH = Path("inference_traffic_light_video.mp4")
WEIGHTS_PATH = Path("runs/best.pt")
MODEL_PATH = Path("yolov8n.pt")
CLASS_NAMES = ["green", "off", "red", "wait_on", "yellow"]

In [ ]:
import yaml
TRAIN_IMAGES_DIR = DATA_ROOT / "train" / "images"
VAL_IMAGES_DIR = DATA_ROOT / "valid" / "images"
if not VAL_IMAGES_DIR.exists():
    alt = DATA_ROOT / "val" / "images"
    if alt.exists():
        VAL_IMAGES_DIR = alt
if not TRAIN_IMAGES_DIR.exists():
    raise FileNotFoundError(f"Missing training images in {TRAIN_IMAGES_DIR}")
if not VAL_IMAGES_DIR.exists():
    raise FileNotFoundError(f"Missing validation images in {VAL_IMAGES_DIR}")
cfg = {
    "path": str(DATA_ROOT.resolve()),
    "train": str(TRAIN_IMAGES_DIR.resolve()),
    "val": str(VAL_IMAGES_DIR.resolve()),
    "names": dict(enumerate(CLASS_NAMES)),
}
out_dir = Path("runs/data_configs"); out_dir.mkdir(parents=True, exist_ok=True)
DATA_YAML_PATH = out_dir / f"traffic_light_{DATA_ROOT.name.replace(' ', '_')}.yaml"
content = yaml.safe_dump(cfg, sort_keys=False)
if not DATA_YAML_PATH.exists() or DATA_YAML_PATH.read_text() != content:
    DATA_YAML_PATH.write_text(content)
print(f"Data YAML: {DATA_YAML_PATH}
Train images: {TRAIN_IMAGES_DIR}
Val images: {VAL_IMAGES_DIR}
Classes: {CLASS_NAMES}")

In [ ]:
import random

def yolo_to_xyxy(row, w, h):
    cls, xc, yc, bw, bh = row
    x1 = max(0, int((xc - bw / 2) * w))
    y1 = max(0, int((yc - bh / 2) * h))
    x2 = min(w - 1, int((xc + bw / 2) * w))
    y2 = min(h - 1, int((yc + bh / 2) * h))
    return int(cls), x1, y1, x2, y2

candidates = sorted(TRAIN_IMAGES_DIR.glob("*.jpg")) + sorted(TRAIN_IMAGES_DIR.glob("*.png"))
if not candidates:
    print("No training images found for preview.")
else:
    picks = random.sample(candidates, min(3, len(candidates)))
    fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5))
    axes = [axes] if len(picks) == 1 else axes
    for ax, img_path in zip(axes, picks):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                cls, x1, y1, x2, y2 = yolo_to_xyxy([float(x) for x in line.split()], w, h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                ax.text(x1, max(12, y1 + 12), CLASS_NAMES[cls], color="yellow", fontsize=9,
                        bbox=dict(facecolor="black", alpha=0.5, pad=1))
        ax.imshow(img); ax.set_title(img_path.name); ax.axis("off")
    plt.show()

In [ ]:
import subprocess, sys

train_cmd = [
    sys.executable,
    "Project_1_object_detection_traffic_light.py",
    "train",
    "--data-root", str(DATA_ROOT),
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--seed", str(SEED),
    "--patience", "20",
    "--model", str(MODEL_PATH),
]
result = subprocess.run(train_cmd, check=False)
if result.returncode != 0:
    raise SystemExit(f"Train command failed with exit code {result.returncode}")

In [ ]:
import subprocess, sys

validate_cmd = [
    sys.executable,
    "Project_1_object_detection_traffic_light.py",
    "validate",
    "--data-root", str(DATA_ROOT),
    "--weights", str(WEIGHTS_PATH),
]
result = subprocess.run(validate_cmd, check=False)
if result.returncode != 0:
    raise SystemExit(f"Validate command failed with exit code {result.returncode}")

In [ ]:
import subprocess, sys

infer_cmd = [
    sys.executable,
    "Project_1_object_detection_traffic_light.py",
    "infer-video",
    "--weights", str(WEIGHTS_PATH),
    "--video", str(VIDEO_PATH),
    "--sahi", str(USE_SAHI).lower(),
    "--conf-thres", "0.25",
    "--iou-thres", "0.5",
]
result = subprocess.run(infer_cmd, check=False)
if result.returncode != 0:
    raise SystemExit(f"Inference failed with exit code {result.returncode}")

In [ ]:
import json
from pathlib import Path

metrics_files = sorted(Path("runs").rglob("metrics.json"), key=lambda p: p.stat().st_mtime)
if metrics_files:
    latest = metrics_files[-1]
    with latest.open() as fh:
        metrics = json.load(fh)
    print("Latest metrics file:", latest)
    print("mAP50-95:", metrics.get("map50-95"))
else:
    print("No metrics.json files found. Train the model first.")
print("Annotated video:", Path("outputs/annotated.mp4"))
print("Detections CSV:", Path("outputs/detections.csv"))
print("Frame summary:", Path("outputs/summary_per_frame.csv"))

In [ ]:
if "cv2" not in globals() or cv2 is None:
    print("OpenCV not available. Install `opencv-python` to preview frames.")
else:
    import matplotlib.pyplot as plt

    video_path = Path("outputs/annotated.mp4")
    if not video_path.exists():
        print("Annotated video not found. Run the inference step first.")
    else:
        cap = cv2.VideoCapture(str(video_path))
        frames = []
        for _ in range(3):
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if not frames:
            print("No frames available for preview.")
        else:
            fig, axes = plt.subplots(1, len(frames), figsize=(15, 5))
            if len(frames) == 1:
                axes = [axes]
            for ax, frame in zip(axes, frames):
                ax.imshow(frame)
                ax.axis("off")
            plt.show()

### Notes & Next steps

- Explore larger YOLO backbones or RT-DETR for higher accuracy.
- Extend augmentation strategies (colour jitter, histogram equalisation).
- Investigate mixed-precision training once GPU acceleration is available.